In [3]:
# dictionary comprehension
dict={x:x**2 for x in range(0,50) }
print(dict)
list1=[x**2 for x in range(0,11)]
print(list1)

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16, 5: 25, 6: 36, 7: 49, 8: 64, 9: 81, 10: 100, 11: 121, 12: 144, 13: 169, 14: 196, 15: 225, 16: 256, 17: 289, 18: 324, 19: 361, 20: 400, 21: 441, 22: 484, 23: 529, 24: 576, 25: 625, 26: 676, 27: 729, 28: 784, 29: 841, 30: 900, 31: 961, 32: 1024, 33: 1089, 34: 1156, 35: 1225, 36: 1296, 37: 1369, 38: 1444, 39: 1521, 40: 1600, 41: 1681, 42: 1764, 43: 1849, 44: 1936, 45: 2025, 46: 2116, 47: 2209, 48: 2304, 49: 2401}
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81, 100]


# Pandas Core Basics: Deep Architectural & Practical Introduction

### Core Topics Covered:
1. **Pandas Architecture & NumPy Foundation**: Why Pandas exists, when to use it, and how it compares to Python lists, dicts, and NumPy.
2. **The Pandas Series**: Under-the-hood analysis of 1D labeled arrays, memory mapping, and attributes.
3. **The Pandas DataFrame**: Tabular representation of multi-column data, and memory storage layouts (BlockManager vs. ArrayManager).
4. **Data Loading Mechanics**: A deep dive into the arguments of `read_csv` and `read_excel`, and the `.squeeze()` operation.
5. **Data Exploration & Schema Inspection**: Understanding shape, index/column metadata, `.info()`, `.describe()`, and `.value_counts()`.

<div style="padding: 15px; border-left: 5px solid #2196F3; background-color: #e3f2fd; border-radius: 4px; margin-bottom: 10px;">
<strong>Prerequisites:</strong> Ensure you have the local files <code>subs.csv</code>, <code>kohli_ipl.csv</code>, <code>bollywood.csv</code>, and <code>sales_data.xlsx</code> in your current workspace directory.
</div>


## Section 1: NumPy vs. Pandas: The Core Paradigm Shift

Before writing code, we must understand *why* we use Pandas instead of raw Python or NumPy. 

### 1. The Limitations of Raw Python
* **Lists:** A Python list is a collection of pointers to objects. Because elements can reside anywhere in memory, lists have high overhead (boxing/unboxing) and do not support vectorized mathematical operations. Running statistical operations on a list of 10 million elements in a `for` loop is extremely slow.
* **Dictionaries:** Dictionaries offer fast $O(1)$ key lookups, but they lack tabular capabilities (like 2D slicing, matrix mathematics, column-wise aggregations, and visual grid presentation).

### 2. The Limitations of NumPy
* **Homogeneity:** A NumPy `ndarray` is designed for contiguous, homogeneous data. Every element in the array must share the exact same data type (e.g., all 64-bit floats). If you insert a string into a float array, NumPy will coerce the entire array to strings, breaking math functionality.
* **No Label Support:** NumPy arrays are indexed implicitly by integer positions (`0, 1, 2...`). Real-world datasets have columns (like "Name", "Date", "Salary") and row labels (like match numbers, timestamps, or student IDs). Managing these mapping translations manually in NumPy is error-prone.

### 3. The Pandas Solution
Pandas bridges this gap by wrapping NumPy arrays in a **labeled axis shell**. It allows:
* **Heterogeneous Data:** Storing integers, floats, strings, and dates side-by-side (each column can have its own type).
* **Explicit Alignment:** When you add two Pandas structures, they automatically align by their labels, not just positions.
* **Missing Value Handling:** Native support for `NaN` (Not a Number) and clean methods to replace or discard missing data.

### Architectural Comparison Table

| Feature | Python List | NumPy ndarray | Pandas Series / DataFrame |
| :--- | :--- | :--- | :--- |
| **Dimensionality** | 1D (nested for multi-D) | N-Dimensional | 1D (Series), 2D (DataFrame) |
| **Data Types** | Heterogeneous (slow) | Homogeneous (extremely fast) | Heterogeneous across columns |
| **Indexing** | Implicit Integer (0, 1...) | Implicit Integer | Explicit Labeled & Integer |
| **Memory Layout** | Array of pointers (non-contiguous) | Contiguous blocks in memory | Contiguous blocks (BlockManager) |
| **Missing Data** | `None` (manual check) | `np.nan` (no native support) | Built-in NaN alignment & drop/fill helpers |


## Setup & Library Imports

Run this cell to import the Pandas and NumPy libraries. 

We import them with the standard aliases `pd` and `np`. These aliases are a global convention in data science. They prevent namespace clutter (e.g. typing `pandas.Series` vs `pd.Series`) and reduce code keystrokes.


In [3]:
# !pip install pandas
import pandas as pd
import numpy as np

print('Pandas version:', pd.__version__)
print('NumPy version:', np.__version__)


Pandas version: 2.3.2
NumPy version: 2.2.6


## Section 2: Deep-Dive: Pandas Series (1D Labeled Array)

A **Series** is a one-dimensional labeled array. Think of it as a single column of data where every row has a label. 

### The Core Equation:
$$\\text{Series} = \\text{Index (Axis Labels)} + \\text{Values (NumPy Array)} + \\text{Metadata (Name, Dtype)}$$

### The Dual-Nature of Series
A Pandas Series behaves like two structures at the same time:
1. **Array-like:** It supports ordered sequence indexing, slicing, vector operations, and element-wise math (just like a NumPy array).
2. **Dict-like:** It maps index labels to values, allowing fast lookups using keys (just like a Python dictionary).



### Constructor Parameters:
`pd.Series(data=None, index=None, dtype=None, name=None)`
* **`data`**: Any iterable (list, tuple), dictionary, NumPy array, or scalar.
* **`index`**: Custom row labels. Must match the length of `data`. Defaults to `RangeIndex(0, 1, 2, ...)` if not specified.
* **`dtype`**: Data type of values. If None, it will be automatically inferred.
* **`name`**: Metadata representing the name of the Series. When a Series is converted to a DataFrame, this name becomes the column label.


In [8]:
# # 1. Creating a Series from a list (with a custom label index)
marks = [67, 57, 89, 100]
subjects = ['Maths', 'English', 'Science', 'Hindi']

marks_series = pd.Series(marks,index=subjects,dtype='Int16',name="student marks")
print("--- Series from List ---")
print(marks_series)

# # 2. Creating a Series from a Python Dictionary
# Note: Dictionary keys automatically become the Index; values become the Series values.
marks_dict = {
    'Maths': 67,
    'English': 57,
    'Science': 89,
    'Hindi': 100
}
marks_series_dict = pd.Series(marks_dict, name='Student Marks (Dict)')
print("\n--- Series from Dictionary ---")
print(marks_series_dict)


--- Series from List ---
Maths       67
English     57
Science     89
Hindi      100
Name: student marks, dtype: Int16

--- Series from Dictionary ---
Maths       67
English     57
Science     89
Hindi      100
Name: Student Marks (Dict), dtype: int64


### Python Variable Name vs. Series Name (Metadata)

It is crucial to distinguish between:
* **Variable Name (`marks_series`):** The Python pointer/reference in code.
* **Series Name (`'Student Marks'`):** Metadata stored within the Series object. When we join Series into a DataFrame, the Series Name becomes the Column Label!

### Attributes vs. Methods

* **Attributes:** Properties that describe the object. They do not run computations, so they do **not** use parentheses `()`
  * Examples: `.size`, `.dtype`, `.name`, `.index`, `.values`, `.is_unique`.
* **Methods:** Operations that perform actions or computations. They **must** be called with parentheses `()`.
  * Examples: `.head()`, `.tail()`, `.mean()`.


In [9]:
# Inspecting Series Attributes 
print("Size (number of elements):", marks_series.size)
print("Data Type (dtype):", marks_series.dtype)
print("Series Name:", marks_series.name)
print("Is Unique (are all values different?):", marks_series.is_unique)
print("Index Object:", marks_series.index)
print("Values (wrapped NumPy array):", marks_series.values)
print("Type of values:", type(marks_series.values))


Size (number of elements): 4
Data Type (dtype): Int16
Series Name: student marks
Is Unique (are all values different?): True
Index Object: Index(['Maths', 'English', 'Science', 'Hindi'], dtype='str')
Values (wrapped NumPy array): <IntegerArray>
[67, 57, 89, 100]
Length: 4, dtype: Int16
Type of values: <class 'pandas.arrays.IntegerArray'>


## Section 3: Deep-Dive: Pandas DataFrame (2D Tabular Data)

A **DataFrame** is a two-dimensional tabular data structure with labeled axes (rows and columns). It is essentially a collection of **Series** sharing the same index.

### The Core Equation:
$$\\text{DataFrame} = \\text{Series 1} + \\text{Series 2} + \\dots + \\text{Series N}$$

### Tabular Grid Layout:
* **Index (Axis 0):** Row labels.
* **Columns (Axis 1):** Column labels.

### Memory Layout: BlockManager vs. ArrayManager
By default, Pandas uses a **BlockManager** to arrange data in memory:
* **Group-by-Dtype Storage:** Columns of the same data type are stored together in single 2D NumPy arrays (blocks). For example, if a DataFrame has three `int64` columns and two `float64` columns, the BlockManager stores them as one 2D integer array and one 2D float array.
* **Pros:** Highly efficient for vectorized math operations across columns of the same type (like matrix operations).



In [10]:
# Creating a DataFrame from a dictionary of lists
student_data = {
    'Name': ['Aman', 'Ravi', 'Neha', 'Pooja'],
    'Marks': [80, 75, 90, 88],
    'Package (LPA)': [5.5, 4.0, 7.2, 6.0]
}

df_students = pd.DataFrame(student_data)
print("--- Students DataFrame ---")
print(df_students)

# # Attributes of DataFrame
print("\nDataFrame Index (row labels):", df_students.index)
print("DataFrame Columns (column labels):", df_students.columns)
print("DataFrame Shape (rows, columns):", df_students.shape)
print("DataFrame Values:\n", df_students.values)


--- Students DataFrame ---
    Name  Marks  Package (LPA)
0   Aman     80            5.5
1   Ravi     75            4.0
2   Neha     90            7.2
3  Pooja     88            6.0

DataFrame Index (row labels): RangeIndex(start=0, stop=4, step=1)
DataFrame Columns (column labels): Index(['Name', 'Marks', 'Package (LPA)'], dtype='str')
DataFrame Shape (rows, columns): (4, 3)
DataFrame Values:
 [['Aman' 80 5.5]
 ['Ravi' 75 4.0]
 ['Neha' 90 7.2]
 ['Pooja' 88 6.0]]


## Section 4: Loading Datasets: Internal Mechanisms

In real-world data science, we rarely write data manually. Instead, we load tabular files using built-in reader functions like `pd.read_csv()` and `pd.read_excel()`.

### Deep Dive into Reader Arguments:
1. **`filepath_or_buffer`**: Path to the file on disk (or a valid URL string).
2. **`sep` / `delimiter`**: Character separating columns (defaults to `,` for CSV).
3. **`header`**: Row number(s) to use as column names. Defaults to `0` (first row). Use `header=None` if the file has no headers.
4. **`names`**: List of column names to use if the file has no headers or to rename existing columns.
5. **`index_col`**: Column to use as row labels (index) of the DataFrame.
6. **`usecols`**: List of column names or index positions to load. Crucial for memory optimization (e.g. only load columns `['match_no', 'runs']`).
7. **`dtype`**: Dictionary specifying data types for columns (e.g. `{'match_no': np.int32}`).
8. **`na_values`**: Custom string values to recognize as nulls (e.g. `['missing', 'NA', 'N/A', '-999']`).
9. **`parse_dates`**: List of columns to parse as datetime objects.
10. **`nrows`**: Number of rows to read. Useful for testing huge files.
11. **`chunksize`**: Number of rows per chunk for iterator. Allows loading files in batches (out-of-core computing).

### The Squeeze Operation (`.squeeze()`)
When reading a CSV with only one column, Pandas still reads it as a 2D DataFrame. By chaining `.squeeze('columns')` at the end, we convert it into a 1D Series.


In [5]:
# 1. Loading a CSV file as a Series (YouTube Subscribers)
# Since subs.csv has only 1 column, we squeeze it to get a Series.
import pandas as pd
# subs = pd.read_csv('DataSet_used/subs.csv').squeeze('columns')
# print('subs type:', type(subs))
# print('subs head:')
# print(subs.head(3))

# 2. Loading a CSV and setting a custom index column (Kohli IPL runs)
# Squeezing it because after setting 'match_no' as the index, only 1 column ('runs') remains.
# vk = pd.read_csv('DataSet_used/kohli_ipl.csv', index_col='match_no').squeeze('columns')
# print('\nvk type:', type(vk))
# print('vk head:')
# print(vk.head(3))

# 3. Loading an Excel file as a DataFrame (Sales Transactions)
sales = pd.read_excel('DataSet_used/sales_data_messy.xlsx')
print('\nsales type:', type(sales))
print('sales shape:', sales.shape)
print('sales head:')
print(sales.head())



sales type: <class 'pandas.DataFrame'>
sales shape: (110, 7)
sales head:
  Transaction_ID        Date     Product     Category  Price  Quantity  \
0       TXN-1001  2026-01-07  Headphones  Accessories  150.0         2   
1       TXN-1002  2026-01-20  Smartphone  Electronics  600.0         4   
2       TXN-1003  2026-01-29  Smartphone  Electronics  600.0         3   
3       TXN-1004  2026-01-15  Headphones  Accessories  150.0         1   
4       TXN-1005  2026-01-11  Headphones  Accessories  150.0         4   

   Revenue  
0      300  
1     2400  
2     1800  
3      150  
4      600  


## Section 5: DataFrame Indexing and Slicing (`loc` vs `iloc`)

Indexing in Pandas allows you to select specific rows and columns from a DataFrame. The two primary methods are `.loc` and `.iloc`.

### 1. Label-based Indexing: `.loc`
* **Meaning:** Location based on **labels**.
* **Usage:** `.loc[row_label, column_label]`
* **Slicing:** When slicing with `.loc`, both the start and end labels are **included**.
* **Example:** `df.loc['row1':'row3', 'colA':'colC']`

### 2. Integer-based Indexing: `.iloc`
* **Meaning:** Integer Location based on **positions** (0-indexed).
* **Usage:** `.iloc[row_position, column_position]`
* **Slicing:** When slicing with `.iloc`, the end index is **excluded** (just like standard Python lists).
* **Example:** `df.iloc[0:3, 0:2]`


In [8]:
# Let's create a sample DataFrame to demonstrate loc and iloc
import pandas as pd

data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'Age': [24, 27, 22, 32, 29],
    'City': ['New York', 'London', 'Paris', 'Berlin', 'Tokyo'],
    'Score': [85.5, 90.0, 88.5, 92.0, 95.5]
}
df_demo = pd.DataFrame(data, index=['A1', 'A2', 'A3', 'A4', 'A5'])
print("--- Original DataFrame ---")
print(df_demo)

# Using .loc (Label-based)
print("\n--- Using .loc: Rows A2 to A4, Columns Name to City ---")
print(df_demo.loc['A2':'A4', 'Name':'City'])

# # Using .iloc (Integer-based)
print("\n--- Using .iloc: Rows 1 to 4, Columns 0 to 2 ---")
print(df_demo.iloc[1:4, 0:2])


--- Original DataFrame ---
       Name  Age      City  Score
A1    Alice   24  New York   85.5
A2      Bob   27    London   90.0
A3  Charlie   22     Paris   88.5
A4    David   32    Berlin   92.0
A5      Eve   29     Tokyo   95.5

--- Using .loc: Rows A2 to A4, Columns Name to City ---
       Name  Age    City
A2      Bob   27  London
A3  Charlie   22   Paris
A4    David   32  Berlin

--- Using .iloc: Rows 1 to 4, Columns 0 to 2 ---
       Name  Age
A2      Bob   27
A3  Charlie   22
A4    David   32


## Section 6: Interactive Practice Exercises

Complete the code tasks below to practice what you have learned.

1. **Exercise 1**: Create a Pandas Series called `s_countries` from the list `['India', 'USA', 'Nepal', 'Sri Lanka']`. Give it the name `'Global Countries'`.
2. **Exercise 2**: Load the dataset `bollywood.csv` into a Series called `movies` with the index column set to `'movie'`. Make sure it is loaded as a Series.
3. **Exercise 3**: Display the shape of `movies` and display a random sample of 5 movies.
.
5. **Exercise 5**: Using `df_demo` (from Section 5), use `.loc` to extract the 'Age' and 'Score' of 'Alice' (row 'A1') and 'Eve' (row 'A5').
6. **Exercise 6**: Using `df_demo`, use `.iloc` to extract the first 3 rows and the last 2 columns.


In [ ]:
# 1. Create Series s_countries (Name: 'Global Countries')
countries = ['India', 'USA', 'Nepal', 'Sri Lanka']
s_countries = None
print('s_countries:')
print(s_countries)

# 2. Load bollywood.csv with custom index movie (Squeeze into a Series)
movies = pd.read_csv('DataSet_used/bollywood.csv')

# 3. Display shape and a random sample of 5 movies of the movies Series
# Explain why .sample() is useful here.
movies = pd.read_csv('DataSet_used/bollywood.csv', index_col='movie').squeeze('columns')
movies.sample(5)

# 4. Use .loc to extract Age and Score of A1 and A5 from df_demo
df_demo = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva'],
    'Age': [25, 30, 35, 40, 45],
    'City': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'],
    'Score': [85.5, 90.0, 95.5, 80.0, 75.5]
}, index = ['A1', 'A2', 'A3', 'A4', 'A5'])
df_demo.head()
df_demo.iloc[0, 1:4:2], df_demo.iloc[-1, 1:4:2]

# 6. Use .iloc to extract first 3 rows and last 2 columns from df_demo
df_demo.iloc[0:3, -2:]



s_countries:
None

movies type: <class 'NoneType'>


,City,Score
A1,New York,85.5
A2,Los Angeles,90.0
A3,Chicago,95.5


## Section 7: Important Data Manipulation Functions

* **Column selection**: `df['col']` (returns a Series), `df[['col1', 'col2']]` (returns a DataFrame).

* **Boolean filtering**: `df[df['col'] > 0]` filters rows based on a condition.

* **Multiple conditions**: Use `&` for AND, `|` for OR (ensure conditions are in parentheses, e.g., `df[(df['col1'] > 0) & (df['col2'] == 1)]`).

* **`isin()`**: Filter rows where a column's values match a list, e.g., `df[df['col'].isin([1, 2, 3])]`.

* **`between()`**: Filter rows where a column's values are within a range, e.g., `df[df['col'].between(10, 20)]`.

* **`sort_values()`**: Sort the DataFrame, e.g., `df.sort_values(by='col', ascending=False)`.

* **`groupby()`**: Group data by one or more columns for aggregation, e.g., `df.groupby('category')`.

* **`agg()`**: Apply one or more aggregation functions, e.g., `df.groupby('col').agg({'val': ['sum', 'mean']})`.

* **`pivot_table()`**: Create a spreadsheet-style pivot table, e.g., `df.pivot_table(values='val', index='row', columns='col', aggfunc='mean')`.

* **`crosstab()`**: Compute a simple cross-tabulation of two (or more) factors, e.g., `pd.crosstab(df['col1'], df['col2'])`.

* **`merge()`**: Join two DataFrames on a key, e.g., `pd.merge(df1, df2, on='key', how='inner')`.

* **`concat()`**: Concatenate DataFrames along rows or columns, e.g., `pd.concat([df1, df2], axis=0)`.

* **`apply()`**: Apply a function along an axis of the DataFrame or to a Series, e.g., `df['col'].apply(lambda x: x*2)`.

* **`map()`**: Map values of a Series according to input correspondence, e.g., `df['col'].map({'A': 1, 'B': 2})`.


In [ ]:
# Creating sample DataFrames for demonstration
import pandas as pd
import numpy as np

# Sample Data
data = {
    'Employee': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank'],
    'Department': ['HR', 'IT', 'IT', 'HR', 'Finance', 'Finance'],
    'Salary': [50000, 70000, 65000, 55000, 80000, 75000],
    'Experience (Years)': [2, 5, 4, 3, 7, 6],
    'Performance_Score': [85, 92, 88, 75, 95, 80]
}
df = pd.DataFrame(data)
print("--- Original DataFrame ---\n", df)

# 1. Column selection
print("\n--- Column Selection ---")
print("Single Column (Series):\n", df['Employee'].head(2))
print("Multiple Columns (DataFrame):\n", df[['Employee', 'Salary']].head(2))

# 2. Boolean filtering & 3. Multiple conditions
print("\n--- Boolean Filtering & Multiple Conditions ---")
print("Salary > 60k:\n", df[df['Salary'] > 60000])
print("IT Dept AND Salary > 65k:\n", df[(df['Department'] == 'IT') & (df['Salary'] > 65000)])

# 4. isin() & 5. between()
print("\n--- isin() and between() ---")
print("Dept in HR or IT:\n", df[df['Department'].isin(['HR', 'IT'])])
print("Experience between 3 and 5 years:\n", df[df['Experience (Years)'].between(3, 5)])

# 6. sort_values()
print("\n--- sort_values() ---")
print("Sort by Salary (Descending):\n", df.sort_values('Salary', ascending=False).head(3))

# 7. groupby() & 8. agg()
print("\n--- groupby() and agg() ---")
print("Average Salary by Dept:\n", df.groupby('Department')['Salary'].mean())
print("Multiple Aggregations:\n", df.groupby('Department').agg({'Salary': ['mean', 'sum'], 'Performance_Score': 'max'}))

# 9. pivot_table()
print("\n--- pivot_table() ---")
print(df.pivot_table(values='Salary', index='Department', aggfunc='mean'))

# 10. crosstab()
print("\n--- crosstab() ---")
# Let's add a random 'Bonus' column for crosstab demonstration
df['Bonus_Eligible'] = ['Yes', 'Yes', 'No', 'No', 'Yes', 'No']
print(pd.crosstab(df['Department'], df['Bonus_Eligible']))

# 11. merge() & 12. concat()
print("\n--- merge() and concat() ---")
df2 = pd.DataFrame({'Employee': ['Alice', 'Bob', 'Eve'], 'City': ['NY', 'SF', 'LA']})
print("Merge (Inner Join):\n", pd.merge(df, df2, on='Employee', how='inner'))

df_extra = pd.DataFrame({'Employee': ['Grace'], 'Department': ['IT'], 'Salary': [90000], 'Experience (Years)': [8], 'Performance_Score': [98], 'Bonus_Eligible': ['Yes']})
print("Concat (Append Row):\n", pd.concat([df, df_extra], ignore_index=True).tail(2))

# 13. apply() & 14. map()
print("\n--- apply() and map() ---")
print("Apply (Salary + 10%):\n", df['Salary'].apply(lambda x: x * 1.1).head(3))
dept_codes = {'HR': 1, 'IT': 2, 'Finance': 3}
print("Map (Dept to Code):\n", df['Department'].map(dept_codes).head(3))


## Summary & Revision Checklist

Verify you understand the function of each term below before moving on:
* **Series vs. DataFrame**: Series is 1D (labels + values), DataFrame is 2D (table of Series).
* **NumPy Foundation**: Pandas uses NumPy arrays under the hood to store actual values for high performance.
* **Why Read Squeeze?**: `.squeeze('columns')` turns a 1-column DataFrame into a Series.
* **Attributes vs. Methods**: Attributes do not run calculations (no `()`), methods run code (must use `()`).
* **Attributes list**:
  * `size`: total items.
  * `dtype`: data type of values.
  * `name`: name of Series.
  * `index`: row labels.
  * `values`: values as NumPy array.
* **Methods list**:
  * `head()`: first rows.
  * `tail()`: last rows.
  * `sample()`: random rows (avoids order bias).
  * `info()`: schema types, nulls count, memory.
  * `describe()`: general numeric analytics summary.
  * `value_counts()`: frequency of unique category values.
